# Решения: Практика: воспроизводимый preprocessing pipeline

**Для преподавателя.** Секционный эталон урока и ДЗ; до сдачи ученикам не показывать.


Работаем как команда CRM маркетплейса: из трёх связанных таблиц нужно получить
объяснимые признаки, а не просто добиться вывода без ошибки. Перед каждой
операцией сформулируйте единицу наблюдения, ключ соединения и ожидаемое число
строк. После операции прочитайте assert как исполняемый контракт.

Сначала сделайте минимальный рабочий вариант, затем проверьте его на данных и
только после этого интерпретируйте результат. Не вводите метку churn: в этом
модуле мы строим и проверяем признаки, но не обучаем модель оттока.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_csv(name):
    for path in (
        Path(name),
        Path("../") / name,
        Path("../../data") / name,
        Path("../data") / name,
        Path("../../../data") / name,
    ):
        if path.exists():
            return path.resolve()
    return "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_05_shop_feature_engineering/data/" + name

orders = pd.read_csv(find_csv("orders_slim.csv"), parse_dates=["order_purchase_timestamp", "order_delivered_customer_date"])
customers = pd.read_csv(find_csv("customers_slim.csv"))
payments = pd.read_csv(find_csv("payments_slim.csv"))
assert len(orders) and len(customers) and len(payments)
assert orders["order_id"].is_unique and customers["customer_id"].is_unique
print(f"orders={len(orders)}, customers={len(customers)}, payments={len(payments)}")


## Урок. 1. Контракт функции

In [ ]:
def preprocess_customers(orders_df,customers_df,payments_df):
    '''Return customer features and ordered audit log.'''
    return None,[]
assert preprocess_customers.__doc__


## Урок. 2. Копии и валидация

In [ ]:
def preprocess_customers(orders_df, customers_df, payments_df):
    log=[]
    o,c,p=orders_df.copy(),customers_df.copy(),payments_df.copy()
    required_o={"order_id","customer_id","order_purchase_timestamp","order_delivered_customer_date"}
    required_c={"customer_id","customer_state"}
    required_p={"order_id","payment_type","payment_value"}
    for frame,required,name in ((o,required_o,"orders"),(c,required_c,"customers"),(p,required_p,"payments")):
        missing=required-set(frame.columns)
        if missing: raise KeyError(f"{name} missing columns: {sorted(missing)}")
    if o["order_id"].duplicated().any() or p["order_id"].duplicated().any(): raise ValueError("duplicate order_id")
    if o["order_purchase_timestamp"].isna().any(): raise ValueError("missing order_purchase_timestamp")
    if p["payment_value"].isna().any() or p["payment_value"].lt(0).any(): raise ValueError("invalid payment_value")
    log.append("validated input contracts")
    x=o.merge(p,on="order_id",validate="one_to_one").merge(c[["customer_id","customer_state"]],on="customer_id",validate="many_to_one")
    log.append("merged orders, payments, customers")
    x["days_to_deliver"]=(x["order_delivered_customer_date"]-x["order_purchase_timestamp"]).dt.days
    x["is_card"]=(x["payment_type"]=="credit_card").astype(int)
    ref=x["order_purchase_timestamp"].max()
    features=x.groupby("customer_id").agg(last_purchase=("order_purchase_timestamp","max"),Frequency=("order_id","nunique"),Monetary=("payment_value","sum"),share_card=("is_card","mean"),avg_days_to_deliver=("days_to_deliver","mean"),customer_state=("customer_state","first")).reset_index()
    features["Recency"]=(ref-features["last_purchase"]).dt.days
    features=features[["customer_id","Recency","Frequency","Monetary","share_card","avg_days_to_deliver","customer_state"]]
    log.append("built customer RFM plus features")
    return features,log

features,log=preprocess_customers(orders,customers,payments)
assert log[0]=="validated input contracts"


## Урок. 3. Форма результата

In [ ]:
required={"customer_id","Recency","Frequency","Monetary","share_card","avg_days_to_deliver","customer_state"}
assert required==set(features.columns) and len(features)==778


## Урок. 4. Инварианты RFM

In [ ]:
checks={"orders":int(features["Frequency"].sum())==len(orders),"money":np.isclose(features["Monetary"].sum(),payments["payment_value"].sum()),"recency":features["Recency"].ge(0).all(),"share_card":features["share_card"].between(0,1).all()}
assert set(checks.values())=={True}


## Урок. 5. Негативный тест

In [ ]:
bad=payments.copy(); bad.loc[bad.index[0],"payment_value"]=-10
caught=""
try: preprocess_customers(orders,customers,bad)
except ValueError as e: caught=str(e)
assert "payment_value" in caught


## Урок. 6. Повторяемость

In [ ]:
features2,log2=preprocess_customers(orders,customers,payments)
same=bool(features.equals(features2))
assert same and log2==log


## Урок. 7. Preview артефакта

In [ ]:
preview_path=Path("features_preview.csv")
features.head(25).to_csv(preview_path,index=False)
loaded_preview=pd.read_csv(preview_path)
assert len(loaded_preview)==25


## Урок. 8. Acceptance и отчёт

In [ ]:
acceptance={"contract":log[0].startswith("validated"),"rfm":{"Recency","Frequency","Monetary"}<=set(features),"extras":{"share_card","avg_days_to_deliver"}<=set(features),"audit":len(log)>=3,"preview":preview_path.exists()}
REPORT=f"Pipeline обработал {len(orders)} заказов и получил {len(features)} клиентских строк. Контракт проверяет схему, ключи, даты и оплаты до join. Выход содержит RFM, долю card, средний срок доставки и штат. Лог фиксирует порядок шагов, preview сохраняет проверяемый срез. Результат описывает историю покупок и готов для следующего этапа, но сам не является прогнозом или доказательством поведения."
assert set(acceptance.values())=={True} and len(REPORT)>=320


## ДЗ. 1. Повторная реализация

In [ ]:
def preprocess_customers(orders_df, customers_df, payments_df):
    log=[]
    o,c,p=orders_df.copy(),customers_df.copy(),payments_df.copy()
    required_o={"order_id","customer_id","order_purchase_timestamp","order_delivered_customer_date"}
    required_c={"customer_id","customer_state"}
    required_p={"order_id","payment_type","payment_value"}
    for frame,required,name in ((o,required_o,"orders"),(c,required_c,"customers"),(p,required_p,"payments")):
        missing=required-set(frame.columns)
        if missing: raise KeyError(f"{name} missing columns: {sorted(missing)}")
    if o["order_id"].duplicated().any() or p["order_id"].duplicated().any(): raise ValueError("duplicate order_id")
    if o["order_purchase_timestamp"].isna().any(): raise ValueError("missing order_purchase_timestamp")
    if p["payment_value"].isna().any() or p["payment_value"].lt(0).any(): raise ValueError("invalid payment_value")
    log.append("validated input contracts")
    x=o.merge(p,on="order_id",validate="one_to_one").merge(c[["customer_id","customer_state"]],on="customer_id",validate="many_to_one")
    log.append("merged orders, payments, customers")
    x["days_to_deliver"]=(x["order_delivered_customer_date"]-x["order_purchase_timestamp"]).dt.days
    x["is_card"]=(x["payment_type"]=="credit_card").astype(int)
    ref=x["order_purchase_timestamp"].max()
    features=x.groupby("customer_id").agg(last_purchase=("order_purchase_timestamp","max"),Frequency=("order_id","nunique"),Monetary=("payment_value","sum"),share_card=("is_card","mean"),avg_days_to_deliver=("days_to_deliver","mean"),customer_state=("customer_state","first")).reset_index()
    features["Recency"]=(ref-features["last_purchase"]).dt.days
    features=features[["customer_id","Recency","Frequency","Monetary","share_card","avg_days_to_deliver","customer_state"]]
    log.append("built customer RFM plus features")
    return features,log

features,log=preprocess_customers(orders,customers,payments)
assert len(features)==778


## ДЗ. 2. Стабильность при перестановке строк

In [ ]:
shuffled=orders.sample(frac=1,random_state=55).reset_index(drop=True)
f2,_=preprocess_customers(shuffled,customers,payments)
cols=["customer_id","Recency","Frequency","Monetary","share_card","avg_days_to_deliver","customer_state"]
left=features.sort_values("customer_id")[cols].reset_index(drop=True)
right=f2.sort_values("customer_id")[cols].reset_index(drop=True)
stable=bool(left[["customer_id","customer_state"]].equals(right[["customer_id","customer_state"]]) and np.allclose(left[["Recency","Frequency","Monetary","share_card","avg_days_to_deliver"]],right[["Recency","Frequency","Monetary","share_card","avg_days_to_deliver"]],equal_nan=True))
assert stable


## ДЗ. 3. Quality gate

In [ ]:
gate={"rows":len(features)==778,"unique":features["customer_id"].is_unique,"orders":int(features["Frequency"].sum())==3500,"money":np.isclose(features["Monetary"].sum(),payments["payment_value"].sum()),"log":len(log)>=3,"no_churn":"churn" not in features}
assert len(gate)==6 and set(gate.values())=={True}


## ДЗ. 4. Challenge: параметр ref_date

In [ ]:
def preprocess_at_date(o,c,p,ref_date=None):
    out,log=preprocess_customers(o,c,p); base=o["order_purchase_timestamp"].max()
    ref_date=base if ref_date is None else pd.Timestamp(ref_date)
    if ref_date<base: raise ValueError("ref_date before latest purchase")
    out["Recency"]=out["Recency"]+(ref_date-base).days
    return out,log+[f"used ref_date {ref_date.date()}"]
future=orders["order_purchase_timestamp"].max()+pd.Timedelta(days=7)
f_future,_=preprocess_at_date(orders,customers,payments,future)
assert (f_future["Recency"]==features["Recency"]+7).all()


## ДЗ. 5. Challenge: handoff note

In [ ]:
HANDOFF_NOTE="Вход pipeline — три slim-таблицы заказов, клиентов и оплат с зафиксированными ключами и datetime. Выход — одна строка customer_id с RFM и двумя производными признаками, плюс последовательный лог. Гарантии: схема и ключи проверены, оплаты неотрицательны, суммы и число заказов сохраняются, входы не меняются. Следующий шаг — разделить данные по времени, выбрать задачу и оценить признаки без утечки; обучение модели не входит в этот модуль."
assert len(HANDOFF_NOTE)>=340
